# 01 - Build Indices (API Plan) - AIC 2026

Notebook này xây dựng toàn bộ artifact phục vụ retrieval:

1. Tự động tìm Dataset root trong `/kaggle/input` (không hard-code path).
2. Kiểm tra schema và coverage thực tế của từng modality.
3. Sinh canonical records: bảng keyframe và các bảng đọc cho từng modality.
4. Xây dựng SigLIP2 Visual FAISS Index, BM25 local, Object index và Text-embedding FAISS Index.
5. Ghi `artifact_manifest.json` để NB02 sử dụng.

Toàn bộ logic nằm trong notebook; artifact runtime được ghi vào `/kaggle/working/artifacts`. Mỗi Stage có checkpoint `_done_<Stage>.json` để có thể chạy tiếp an toàn.

In [1]:
# ============================== CELL 0: DEPENDENCIES ==============================
# Kaggle image Không có sẵn FAISS -> Notebook tự cài. Cần Settings -> Internet = On.
# CELL này chạy được cả ở chế độ Save & Run All (Commit), không phụ thuộc thao tác tay.
import importlib, subprocess, sys

def ensure(module, pip_name=None, upgrade=False):
    """Import được thì thôi; không thì pip install rồi Import lại. Trả về True/False."""
    if not upgrade:
        try:
            importlib.import_module(module)
            print("  OK     ", module)
            return True
        except Exception:
            pass
    pkg = pip_name or module
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + (["-U"] if upgrade else []) + [pkg]
    print("  cài    ", pkg, "...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("  [Lỗi] không cài được", pkg, "->", (r.stderr or r.stdout or "")[-400:])
        return False
    importlib.invalidate_caches()
    try:
        importlib.import_module(module)
        print("  OK     ", module, "(vừa cài)")
        return True
    except Exception as e:
        print("  [Lỗi] cài xong vẫn Import lỗi:", e)
        return False

print("kiểm tra thư viện:")
FAISS_OK = ensure("faiss", "faiss-cpu")
for _m in ["pandas", "pyarrow", "scipy", "numpy"]:
    ensure(_m)

if not FAISS_OK:
    print("")
    print("[Cảnh báo] Thiếu FAISS -> không Build/đọc được FAISS Index.")
    print("  1) Settings -> Internet = On")
    print("  2) Chạy lại đúng CELL này")
    print("  3) Nếu vẫn lỗi: thêm một cell và chạy  !pip install -q faiss-cpu")


In [2]:
# ============================== CELL 1: CONFIG ==============================
# CELL config tập trung. Sửa ở đây, không sửa rai rac Trong Notebook.
import os, sys, json, math, re, time, hashlib, unicodedata, shutil, gc, glob
from pathlib import Path

CFG = {
    # --- Data roots. Điền sẵn theo layout thật trên Kaggle của bạn.
    #     Để None để Auto-discover; nếu path dưới đây không tồn tại thì cũng tự động fallback sang đó. ---
    "dataset_root":  "/kaggle/input/datasets/fatle542/AIC-Dataset",
    "feature_root":  "/kaggle/input/datasets/kitnehi1211/feature-AIC-2026/Feature_Dataset",
    "query_root":    "/kaggle/input/datasets/kitnehi1211/dethithunghiem",
    "input_scan_dirs": ["/kaggle/input", "."],
    "scan_depth": 5,           # /kaggle/input/datasets/<owner>/<Dataset>/Feature_Dataset = 4 CAP

    # --- Output ---
    "art_dir": "/kaggle/working/artifacts",
    "cache_dir": "/kaggle/working/cache",
    "force_stages": [],        # vì đủ: ["Audit", "bm25"] để Build lại

    # --- Giới hạn phát triển: đặt số > 0 để chỉ xử lý N Video đầu (Smoke test) ---
    "limit_videos": 0,

    # --- SigLIP2 Visual Index ---
    "siglip_dim": 1536,
    "siglip_renormalize": True,    # README nội đã L2-norm, vấn kiểm tra + renorm lại

    # --- Text documents ---
    "transcript_chunk_max_chars": 480,   # góp segment ASR thành chunk có overlap
    "transcript_chunk_overlap": 1,       # overlap 1 segment
    "summary_chunk_max_chars": 1200,
    "caption_min_chars": 12,
    "ocr_min_conf": 0.35,

    # --- Object Index ---
    "object_conf_threshold": 0.30,
    "object_topk_per_frame": 12,

    # --- BM25 ---
    "bm25_k1": 1.2,
    "bm25_b": 0.75,

    # --- OpenRouter / API ---
    "openrouter_base": "https://openrouter.ai/api/v1",
    "embed_model": "openai/text-embedding-3-small",
    "embed_dim": 1536,
    "embed_batch": 96,
    "embed_max_retries": 4,
    "embed_timeout": 120,
    # Bật/tắt từng nhánh embedding (tắt để tiết kiệm ngân sách)
    "embed_targets": {"caption": True, "transcript_en": True, "summary": True},
    # DRY_RUN=True: chỉ đếm đọc/token và ước tính chỉ phí, Không gọi API.
    "DRY_RUN": False,
    # Hard COST guard (chia sẻ dùng chung với NB02 qua cost_ledger.json).
    "MAX_TOTAL_COST_USD": 2.00,
    "COST_WARN_RATIO": 0.80,
    "COST_HALT_RATIO": 0.90,
    # Giá tham chiếu (USD / 1M token) - Cập nhật nếu OpenRouter đổi Giá.
    "price_embed_per_1m": 0.02,
    # Trần chỉ phí riêng cho Stage embedding của NB01
    "embed_budget_usd": 0.45,
}

ART = Path(CFG["art_dir"]); ART.mkdir(parents=True, exist_ok=True)
CACHE = Path(CFG["cache_dir"]); CACHE.mkdir(parents=True, exist_ok=True)
(CACHE / "embed").mkdir(parents=True, exist_ok=True)

# API key: Chỉ đọc từ env hoặc Kaggle Secret. Không Ghi key thật vào Notebook.
OPENROUTER_API_KEY = ""
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
    except Exception as e:
        print("[warn] không lấy được Kaggle Secret:", type(e).__name__)
print("API key present:", bool(OPENROUTER_API_KEY), "| DRY_RUN:", CFG["DRY_RUN"])
print("artifacts ->", ART)

In [3]:
# ============================== CELL 2: HELPERS ==============================
def stage_done(name):
    return (ART / ("_done_" + name + ".json")).exists() and name not in CFG["force_stages"]

def mark_done(name, info=None):
    (ART / ("_done_" + name + ".json")).write_text(
        json.dumps({"Stage": name, "ts": time.time(), "info": info or {}}, ensure_ascii=False, indent=1),
        encoding="UTF-8")
    print("[Stage:" + name + "] DONE")

def jload(p):
    with open(p, "r", encoding="UTF-8") as f:
        return json.load(f)

def jdump(obj, p):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="UTF-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)

def strip_accents(s):
    s = unicodedata.normalize("NFD", s or "")
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", s).replace("\u0111", "d").replace("\u0110", "D")

def norm_text(s):
    return unicodedata.normalize("NFC", (s or "")).strip()

_TOK = re.compile(r"[0-9a-zA-Z\u00c0-\u1ef9]+", re.UNICODE)
def tokenize(s, fold_accent=False):
    s = norm_text(s).lower()
    if fold_accent:
        s = strip_accents(s)
    return _TOK.findall(s)

def sha1(s):
    return hashlib.sha1(s.encode("UTF-8")).hexdigest()

def human(n):
    n = float(n)
    for u in ["", "K", "M", "G"]:
        if abs(n) < 1000:
            return "%.1f%s" % (n, u)
        n /= 1000.0
    return "%.1fT" % n

print("helpers READY |", tokenize("Th\u1eadp gì\u00e2y 06:30", fold_accent=True))

In [4]:
# ============================== CELL 3: Data DISCOVERY ==============================
# Tự dò Dataset root; không hard-code path Kaggle. Cho phép override qua CFG.
MODALITY_HINTS = {
    "SigLIP":     ["siglip2-features-giant-opt"],
    "clip":       ["clip-features-32-aic25-b1/clip-features-32", "clip-features-32"],
    "caption":    ["Image_captioning"],
    "OCR":        ["OCR_EasyOCR_VietOCR"],
    "summary":    ["Summary_video"],
    "mapkf":      ["map-keyframes-aic25-b1/map-keyframes", "map-keyframes"],
    "media":      ["media-info-aic25-b1/media-info", "media-info"],
    "objects":    ["objects-aic25-b1/objects", "objects"],
    "asr_vi":     ["Transcript_Extract"],
    "asr_en":     ["Transcript_Translated"],
}

# Không duyệt vào nhưng thư mục chắc chặn không phải "root" (tránh liet kế 873 thư mục Keyframe).
_PRUNE = re.compile(r"^(keyframes|Video|Videos_.*|Keyframes_.*|L\d+_V\d+|objects|"
                    r"map-keyframes|media-info|clip-features-32|__pycache__|\..*)$")

def _subdirs(p, depth):
    out = []
    if depth <= 0:
        return out
    try:
        kids = [x for x in sorted(p.iterdir()) if x.is_dir()]
    except Exception:
        return out
    for k in kids:
        out.append(k)
        if not _PRUNE.match(k.name):
            out.extend(_subdirs(k, depth - 1))
    return out

def _candidate_roots():
    roots = []
    for base in CFG["input_scan_dirs"]:
        b = Path(base)
        if not b.exists():
            continue
        roots.append(b)
        roots.extend(_subdirs(b, int(CFG.get("scan_depth", 5))))
    # roots do user chỉ định được ưu tiên
    forced = [CFG.get(k) for k in ("feature_root", "dataset_root", "query_root")]
    pref = [Path(x) for x in forced if x and Path(x).exists()]
    seen, out = set(), []
    for r in pref + roots:
        s = str(r)
        if s not in seen:
            seen.add(s); out.append(r)
    return out

ROOTS = _candidate_roots()

def find_modality_dirs():
    found = {}
    for key, hints in MODALITY_HINTS.items():
        for r in ROOTS:
            for h in hints:
                cand = r / h
                try:
                    if cand.is_dir() and any(cand.iterdir()):
                        found[key] = str(cand); break
                except Exception:
                    pass
            if key in found:
                break
    return found

def find_video_and_keyframe_dirs():
    vids, kfs = {}, {}
    for r in ROOTS:
        try:
            vdirs = sorted(r.glob("Videos_*")) + sorted(r.glob("videos_*"))
            kdirs = sorted(r.glob("Keyframes_*")) + sorted(r.glob("keyframes_*"))
        except Exception:
            continue
        for p in vdirs:
            if not p.is_dir():
                continue
            inner = p / "Video"
            d = inner if inner.is_dir() else p
            for mp4 in d.rglob("*.mp4"):
                vids.setdefault(mp4.stem, str(mp4))
        for p in kdirs:
            if not p.is_dir():
                continue
            inner = p / "keyframes"
            d = inner if inner.is_dir() else p
            for kd in sorted(d.iterdir()):
                if kd.is_dir():
                    kfs.setdefault(kd.name, str(kd))
    return vids, kfs

def find_query_dir():
    if CFG["query_root"]:
        return CFG["query_root"]
    best, best_n = None, 0
    for r in ROOTS:
        try:
            n = len([p for p in r.iterdir() if p.is_file()
                     and p.suffix.lower() == ".txt"
                     and p.stem.lower().startswith("query-")])
        except Exception:
            n = 0
        if n > best_n:
            best, best_n = str(r), n
    return best

MOD = find_modality_dirs()
VIDEO_FILES, KEYFRAME_DIRS = find_video_and_keyframe_dirs()
QUERY_DIR = find_query_dir()

print("--- modality dirs ---")
for k in MODALITY_HINTS:
    print("  %-10s: %s" % (k, MOD.get(k, "*** MISSING ***")))
print("videos found     :", len(VIDEO_FILES))
print("Keyframe dirs    :", len(KEYFRAME_DIRS))
print("Query dir        :", QUERY_DIR)

assert "SigLIP" in MOD, "Không tìm thấy siglip2-features-giant-opt -> không thể Build Visual Index"
assert "mapkf" in MOD, "Không tìm thấy map-keyframes -> không thể map frame_idx"

In [5]:
# ============================== CELL 4: Video LIST + ASR PATH Index ==============================
import pandas as pd, numpy as np

SIG_DIR = Path(MOD["SigLIP"]); MAP_DIR = Path(MOD["mapkf"])
video_ids = sorted(p.stem for p in SIG_DIR.glob("*.npy"))
if CFG["limit_videos"] > 0:
    video_ids = video_ids[: CFG["limit_videos"]]
print("videos (SigLIP):", len(video_ids))

# ASR nằm trong nhiều batch dir (Videos_L21_a ...) -> Build path Index phang
def index_asr(root_key):
    out = {}
    if root_key not in MOD:
        return out
    for p in Path(MOD[root_key]).rglob("*.json"):
        out.setdefault(p.stem, str(p))
    return out

ASR_VI = index_asr("asr_vi")
ASR_EN = index_asr("asr_en")
print("asr_vi Files:", len(ASR_VI), "| asr_en Files:", len(ASR_EN))

def mod_file(key, vid, ext=".json"):
    if key not in MOD:
        return None
    p = Path(MOD[key]) / (vid + ext)
    return str(p) if p.exists() else None

## Stage `Audit` - Kiểm tra schema và coverage

Với mỗi modality, notebook mở mẫu dữ liệu, kiểm tra các field bắt buộc, dtype, dimension và row alignment. Điều kiện quan trọng nhất là `SigLIP.shape[0] == len(map-keyframes)` cho từng video.

Nếu lệch, video được đánh dấu `row_mismatch`; index chỉ sử dụng phần giao nhau `min(n_rows, n_map)` để không làm hỏng toàn bộ pipeline.

In [6]:
# ============================== CELL 5: Stage Audit ==============================
def audit_video(vid):
    r = {"video_id": vid}
    sp = SIG_DIR / (vid + ".npy")
    try:
        a = np.load(sp, mmap_mode="r")
        r["sig_rows"], r["sig_dim"], r["sig_dtype"] = int(a.shape[0]), int(a.shape[1]), str(a.dtype)
        v = np.asarray(a[: min(8, a.shape[0])], dtype=np.float32)
        r["sig_norm_mean"] = float(np.linalg.norm(v, axis=1).mean()) if len(v) else 0.0
    except Exception as e:
        r["sig_rows"] = 0; r["sig_dim"] = 0; r["sig_norm_mean"] = 0.0
        r["error_sig"] = type(e).__name__ + ": " + str(e)

    mp = MAP_DIR / (vid + ".csv")
    r["map_rows"], r["map_ok"] = 0, False
    if mp.exists():
        try:
            m = pd.read_csv(mp)
            r["map_rows"] = int(len(m))
            r["map_cols"] = ",".join(map(str, m.columns))
            r["fps"] = float(m["fps"].iloc[0]) if "fps" in m.columns and len(m) else float("nan")
            r["map_ok"] = set(["N", "pts_time", "fps", "frame_idx"]).issubset(set(m.columns))
        except Exception as e:
            r["error_map"] = str(e)

    r["row_mismatch"] = int(r.get("sig_rows", 0)) != int(r.get("map_rows", 0))

    cap = mod_file("caption", vid)
    r["has_caption"] = bool(cap)
    if cap:
        try:
            d = jload(cap); ks = d.get("keyframes", [])
            r["caption_kf"] = len(ks)
            r["caption_nonempty"] = sum(1 for k in ks if norm_text(k.get("caption")))
        except Exception as e:
            r["error_caption"] = str(e)
    ocr = mod_file("OCR", vid)
    r["has_ocr"] = bool(ocr)
    if ocr:
        try:
            d = jload(ocr); ks = d.get("keyframes", [])
            r["ocr_kf"] = len(ks)
            r["ocr_nonempty"] = sum(1 for k in ks if norm_text(k.get("text")))
        except Exception as e:
            r["error_ocr"] = str(e)
    smy = mod_file("summary", vid)
    r["has_summary"] = bool(smy)
    if smy:
        try:
            r["summary_chars"] = len(norm_text(jload(smy).get("summary", "")))
        except Exception as e:
            r["error_summary"] = str(e)
    r["has_media"] = bool(mod_file("media", vid))
    r["has_asr_vi"] = vid in ASR_VI
    r["has_asr_en"] = vid in ASR_EN
    if vid in ASR_EN:
        try:
            d = jload(ASR_EN[vid]); segs = d.get("segments", [])
            r["asr_segments"] = len(segs)
            r["asr_has_en_seg"] = sum(1 for s in segs if norm_text(s.get("text_en") or ""))
        except Exception as e:
            r["error_asr_en"] = str(e)
    r["has_objects"] = bool("objects" in MOD and (Path(MOD["objects"]) / vid).is_dir())
    r["has_video_file"] = vid in VIDEO_FILES
    r["has_keyframe_dir"] = vid in KEYFRAME_DIRS
    return r

if stage_done("Audit"):
    audit = pd.read_parquet(ART / "audit_videos.parquet")
    print("[Stage:Audit] skip (loaded)", len(audit))
else:
    rows, t0 = [], time.time()
    for i, vid in enumerate(video_ids):
        rows.append(audit_video(vid))
        if (i + 1) % 100 == 0:
            print("  audited %d/%d  %.0fs" % (i + 1, len(video_ids), time.time() - t0))
    audit = pd.DataFrame(rows)
    audit.to_parquet(ART / "audit_videos.parquet", index=False)
    mark_done("Audit", {"n_videos": len(audit)})

n = len(audit)
print("\n===== Audit REPORT =====")
print("videos:", n, "| Keyframe rows (SigLIP):", int(audit["sig_rows"].sum()))
print("SigLIP dims:", sorted(audit["sig_dim"].dropna().unique().tolist()))
print("SigLIP L2-norm mean (trước renorm):", round(float(audit["sig_norm_mean"].mean()), 5))
print("row_mismatch SigLIP vs map:", int(audit["row_mismatch"].sum()), "videos")
for c in ["has_caption", "has_ocr", "has_summary", "has_media", "has_asr_vi",
          "has_asr_en", "has_objects", "has_video_file", "has_keyframe_dir"]:
    print("  coverage %-18s: %4d/%d  %.1f%%" % (c, int(audit[c].sum()), n, 100.0 * audit[c].mean()))
bad = audit[audit["row_mismatch"]][["video_id", "sig_rows", "map_rows"]]
if len(bad):
    print("\n[check] mismatch videos (sẽ dùng min rows):")
    print(bad.head(20).to_string(index=False))
audit.to_csv(ART / "audit_videos.csv", index=False)

## Stage `canonical` - Bảng keyframe chuẩn và các bảng đọc

`keyframes.parquet` là nguồn sự thật duy nhất cho `row_id` của FAISS Visual Index. `row_id` là thứ tự toàn cục, được sinh bằng cách nối các video theo `video_ids` đã sắp xếp và `N` tăng dần.

Các bảng `docs_*.parquet` tuân theo canonical schema, có `video_id`, `keyframe_n`, `frame_idx`, timestamp và modality tương ứng.

In [7]:
# ============================== CELL 6: Stage canonical_kf ==============================
if stage_done("canonical_kf"):
    KF = pd.read_parquet(ART / "keyframes.parquet")
    print("[Stage:canonical_kf] skip (loaded)", len(KF))
else:
    audit_idx = audit.set_index("video_id")
    recs, row_id = [], 0
    for vid in video_ids:
        if vid not in audit_idx.index:
            continue
        a = audit_idx.loc[vid]
        n_rows = int(min(int(a.get("sig_rows", 0) or 0), int(a.get("map_rows", 0) or 0)))
        if n_rows <= 0:
            continue
        m = pd.read_csv(MAP_DIR / (vid + ".csv")).iloc[:n_rows]
        kf_dir = KEYFRAME_DIRS.get(vid)
        for j in range(n_rows):
            nn = int(m["n"].iloc[j])
            recs.append({
                "row_id": row_id, "video_id": vid, "keyframe_n": nn,
                "frame_idx": int(m["frame_idx"].iloc[j]),
                "pts_time": float(m["pts_time"].iloc[j]),
                "fps": float(m["fps"].iloc[j]),
                "kf_path": (str(Path(kf_dir) / ("%03d.jpg" % nn)) if kf_dir else ""),
            })
            row_id += 1
    KF = pd.DataFrame(recs)
    sample = KF.sample(min(200, len(KF)), random_state=0)
    ok = sum(1 for p in sample.kf_path if p and Path(p).exists())
    print("kf_path exists On sample: %d/%d" % (ok, len(sample)))
    if ok < len(sample) * 0.5:
        print("[warn] tên file keyframe có thể khác định dạng %03d.jpg -> kiểm tra lại KEYFRAME_DIRS")
    KF.to_parquet(ART / "keyframes.parquet", index=False)
    mark_done("canonical_kf", {"rows": len(KF)})

KF_INDEX = {(r.video_id, r.keyframe_n): r.row_id for r in KF.itertuples()}
VID_ROWS = KF.groupby("video_id")["row_id"].agg(["min", "max", "count"]).to_dict("index")
KF_BY_VIDEO = {v: g.reset_index(drop=True) for v, g in KF.groupby("video_id")}
print("TOTAL Keyframe rows:", len(KF), "| videos:", KF.video_id.nunique())
print(KF.head(3).to_string(index=False))

In [8]:
# ============================== CELL 7: Stage canonical_docs ==============================
CANON_COLS = ["doc_id", "video_id", "keyframe_n", "frame_idx", "pts_time", "start_time", "end_time",
              "modality", "language", "text", "source_id", "source_score", "is_missing"]

def mk_doc(doc_id, video_id, modality, language, text, keyframe_n=-1, frame_idx=-1,
           pts_time=float("nan"), start_time=float("nan"), end_time=float("nan"),
           source_id="", source_score=float("nan"), is_missing=False):
    return {"doc_id": doc_id, "video_id": video_id, "keyframe_n": int(keyframe_n),
            "frame_idx": int(frame_idx), "pts_time": float(pts_time), "start_time": float(start_time),
            "end_time": float(end_time), "modality": modality, "language": language,
            "text": norm_text(text), "source_id": str(source_id),
            "source_score": float(source_score), "is_missing": bool(is_missing)}

def kf_time(vid, n):
    rid = KF_INDEX.get((vid, n))
    if rid is None:
        return -1, float("nan")
    r = KF.iloc[rid]
    return int(r.frame_idx), float(r.pts_time)

def nearest_kf(vid, t):
    g = KF_BY_VIDEO.get(vid)
    if g is None or not len(g) or not np.isfinite(t):
        return -1, -1
    k = int((g.pts_time - t).abs().values.argmin())
    return int(g.keyframe_n.iloc[k]), int(g.frame_idx.iloc[k])

def build_caption_docs():
    out = []
    for vid in video_ids:
        p = mod_file("caption", vid)
        if not p:
            continue
        try:
            d = jload(p)
        except Exception:
            continue
        for k in d.get("keyframes", []):
            txt = norm_text(k.get("caption"))
            if len(txt) < CFG["caption_min_chars"]:
                continue
            nn = int(k.get("n", -1)); fi, pt = kf_time(vid, nn)
            if fi < 0:
                continue
            out.append(mk_doc("CAP::%s::%d" % (vid, nn), vid, "caption", "EN", txt,
                              nn, fi, pt, pt, pt, source_id=k.get("keyframe", "")))
    return pd.DataFrame(out, columns=CANON_COLS)

def build_ocr_docs():
    out = []
    for vid in video_ids:
        p = mod_file("OCR", vid)
        if not p:
            continue
        try:
            d = jload(p)
        except Exception:
            continue
        for k in d.get("keyframes", []):
            dets = [x for x in (k.get("detections") or [])
                    if x.get("kept", True) and float(x.get("confidence", 0)) >= CFG["ocr_min_conf"]]
            txt = norm_text(k.get("text")) or " ".join(norm_text(x.get("text")) for x in dets)
            if not txt:
                continue
            nn = int(k.get("n", -1)); fi, pt = kf_time(vid, nn)
            if fi < 0:
                continue
            conf = float(np.mean([float(x.get("confidence", 0)) for x in dets])) if dets else float("nan")
            out.append(mk_doc("OCR::%s::%d" % (vid, nn), vid, "ocr", "vi", txt,
                              nn, fi, pt, pt, pt, source_id=k.get("keyframe", ""), source_score=conf))
    return pd.DataFrame(out, columns=CANON_COLS)

def _chunk_segments(segs, key, max_chars, overlap):
    """Góp segment ASR thành chunk có overlap, giữ start/end thật."""
    chunks, cur, cur_len = [], [], 0
    for s in segs:
        t = norm_text(s.get(key) or "")
        if not t:
            continue
        cur.append(s); cur_len += len(t) + 1
        if cur_len >= max_chars:
            chunks.append(list(cur))
            cur = cur[-overlap:] if overlap > 0 else []
            cur_len = sum(len(norm_text(x.get(key) or "")) + 1 for x in cur)
    if cur:
        chunks.append(cur)
    return chunks

def _seg_time(s, key_start, key_end, fallback):
    v = s.get(key_start, s.get(key_end, fallback))
    try:
        return float(v)
    except Exception:
        return float(fallback)

def build_transcript_docs(lang):
    """lang='vi' -> Transcript_Extract (field 'text'); lang='EN' -> Transcript_Translated ('text_en' nếu có)."""
    src_map = ASR_VI if lang == "vi" else ASR_EN
    out = []
    for vid in video_ids:
        p = src_map.get(vid)
        if not p:
            continue
        try:
            d = jload(p)
        except Exception:
            continue
        segs = d.get("segments") or []
        key = "text"
        if lang == "EN" and any(norm_text(s.get("text_en") or "") for s in segs):
            key = "text_en"
        for ci, ch in enumerate(_chunk_segments(segs, key, CFG["transcript_chunk_max_chars"],
                                                CFG["transcript_chunk_overlap"])):
            txt = " ".join(norm_text(s.get(key) or "") for s in ch).strip()
            if not txt:
                continue
            st = _seg_time(ch[0], "start", "start_time", 0.0)
            en = _seg_time(ch[-1], "end", "end_time", st)
            mid = 0.5 * (st + en)
            nn, fi = nearest_kf(vid, mid)
            out.append(mk_doc("ASR%s::%s::%d" % (lang, vid, ci), vid, "transcript_" + lang, lang,
                              txt, nn, fi, mid, st, en, source_id="seg%d" % ci))
    return pd.DataFrame(out, columns=CANON_COLS)

def build_summary_docs():
    out = []
    step = CFG["summary_chunk_max_chars"]
    for vid in video_ids:
        rec_txt = ""
        p = mod_file("summary", vid)
        if p:
            try:
                rec_txt = norm_text(jload(p).get("summary", ""))
            except Exception:
                rec_txt = ""
        meta_bits = []
        mp = mod_file("media", vid)
        if mp:
            try:
                md = jload(mp)
                for f in ["title", "description", "keywords", "author", "publish_date"]:
                    v = md.get(f)
                    if isinstance(v, list):
                        v = " ".join(map(str, v))
                    if v:
                        meta_bits.append("%s: %s" % (f, norm_text(str(v))[:800]))
            except Exception:
                pass
        if rec_txt:
            for ci in range(0, len(rec_txt), step):
                seg = rec_txt[ci:ci + step]
                if seg.strip():
                    out.append(mk_doc("sum::%s::%d" % (vid, ci // step), vid, "summary", "EN",
                                      seg, source_id="summary"))
        if meta_bits:
            out.append(mk_doc("meta::%s" % vid, vid, "metadata", "vi", " | ".join(meta_bits),
                              source_id="media-info"))
        if not rec_txt and not meta_bits:
            out.append(mk_doc("sum::%s::empty" % vid, vid, "summary", "EN", "", is_missing=True))
    return pd.DataFrame(out, columns=CANON_COLS)

DOC_BUILDERS = {
    "caption": build_caption_docs,
    "ocr": build_ocr_docs,
    "transcript_vi": lambda: build_transcript_docs("vi"),
    "transcript_en": lambda: build_transcript_docs("EN"),
    "summary": build_summary_docs,
}

DOCS = {}
for name, fn in DOC_BUILDERS.items():
    fp = ART / ("docs_%s.parquet" % name)
    if fp.exists() and "canonical_docs" not in CFG["force_stages"]:
        DOCS[name] = pd.read_parquet(fp)
        print("  loaded docs_%s: %d" % (name, len(DOCS[name])))
    else:
        t0 = time.time(); df = fn()
        df = df[(df.text.astype(str).str.len() > 0) | df.is_missing].reset_index(drop=True)
        df.to_parquet(fp, index=False); DOCS[name] = df
        print("  built docs_%s: %d rows in %.0fs" % (name, len(df), time.time() - t0))
mark_done("canonical_docs", {k: int(len(v)) for k, v in DOCS.items()})
for k, v in DOCS.items():
    print("%-14s docs=%7d videos=%4d avg_chars=%.0f" % (k, len(v), v.video_id.nunique(), v.text.str.len().mean()))

## Stage `visual` - SigLIP2 Visual FAISS

- Metric là inner product trên vector đã L2-normalize, tương đương cosine similarity.
- Dùng `IndexFlatIP` khi số row còn vừa RAM; nếu vượt `IVF_THRESHOLD` thì dùng `IVFFlat`.
- Index được lưu bền vững trên CPU; NB02 có thể clone sang GPU khi truy vấn.
- `row_id` trong index phải khớp `row_id` trong `keyframes.parquet` và được assert khi kiểm tra.

In [9]:
# ============================== CELL 8: Stage visual_index ==============================
try:
    import faiss
    HAS_FAISS = True
except Exception as e:
    HAS_FAISS = False
    print("[warn] Thiếu FAISS:", e, "-> chạy: !pip install -q faiss-cpu")

IVF_THRESHOLD = 400000   # trên nguong này dùng IVF thấy vì Flat

if stage_done("visual_index"):
    print("[Stage:visual_index] skip")
else:
    assert HAS_FAISS, "Cần FAISS để Build Index"
    N, D = len(KF), CFG["siglip_dim"]
    print("building Visual matrix N=%d D=%d (~%.2f GB float32)" % (N, D, N * D * 4 / 1e9))
    X = np.zeros((N, D), dtype=np.float32)
    cursor, norms = 0, []
    for vid in video_ids:
        info = VID_ROWS.get(vid)
        if not info:
            continue
        cnt = int(info["count"])
        a = np.load(SIG_DIR / (vid + ".npy"), mmap_mode="r")
        v = np.asarray(a[:cnt], dtype=np.float32)
        assert v.shape == (cnt, D), "%s: shape %s != %s" % (vid, v.shape, (cnt, D))
        nrm = np.linalg.norm(v, axis=1, keepdims=True)
        norms.append(float(nrm.mean()))
        if CFG["siglip_renormalize"]:
            v = v / np.clip(nrm, 1e-8, None)
        X[cursor:cursor + cnt] = v
        assert int(info["min"]) == cursor, "row_id lệch tải %s: %s != %s" % (vid, info["min"], cursor)
        cursor += cnt
    assert cursor == N, "cursor %d != N %d" % (cursor, N)
    print("mean L2 norm trước renorm:", round(float(np.mean(norms)), 5))

    if N <= IVF_THRESHOLD:
        index = faiss.IndexFlatIP(D); index.add(X); kind = "FlatIP"
    else:
        nlist = int(max(256, min(65536, 4 * math.sqrt(N))))
        index = faiss.IndexIVFFlat(faiss.IndexFlatIP(D), D, nlist, faiss.METRIC_INNER_PRODUCT)
        tn = min(N, max(50 * nlist, 100000))
        index.train(X[np.random.RandomState(0).choice(N, tn, replace=False)])
        index.add(X); index.nprobe = max(16, nlist // 32); kind = "IVFFlat(nlist=%d)" % nlist
    print("Index:", kind, "| ntotal:", index.ntotal)

    Dq, Iq = index.search(X[:1], 3)
    print("smoke self-search row0 ->", Iq[0].tolist(), np.round(Dq[0], 4).tolist())
    assert Iq[0][0] == 0, "Row alignment sai"

    faiss.write_index(index, str(ART / "faiss_siglip2.index"))
    np.save(ART / "siglip2_matrix_f16.npy", X.astype(np.float16))  # NB02 rescore local nhánh
    mark_done("visual_index", {"kind": kind, "ntotal": int(index.ntotal), "dim": D})
    del X; gc.collect()

## Stage `bm25` - BM25 multi-field local

Notebook tự cài BM25-Okapi trên `scipy.sparse`, nên vẫn chạy được khi Kaggle offline và không cần `rank_bm25`.

Hai biến thể `raw` và `fold` được lưu riêng: `raw` giữ tín hiệu chính xác, còn `fold` hỗ trợ truy vấn thiếu dấu hoặc OCR sai dấu.

In [10]:
# ============================== CELL 9: Stage bm25 ==============================
import scipy.sparse as sp
from collections import Counter, defaultdict

class BM25:
    """BM25-Okapi trên CSR term-document matrix. Persist bằng scipy npz + vocab json."""
    def __init__(self, k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        self.vocab = {}; self.idf = None; self.M = None
        self.dl = None; self.avgdl = 1.0; self._Mcsc = None

    def fit(self, docs_tokens):
        vocab = {}
        indptr, indices, data = [0], [], []
        dl = np.zeros(len(docs_tokens), dtype=np.float32)
        for i, toks in enumerate(docs_tokens):
            c = Counter(toks); dl[i] = len(toks)
            for t, f in c.items():
                j = vocab.get(t)
                if j is None:
                    j = len(vocab); vocab[t] = j
                indices.append(j); data.append(f)
            indptr.append(len(indices))
        V, Nd = len(vocab), len(docs_tokens)
        self.M = sp.csr_matrix((np.asarray(data, np.float32), np.asarray(indices, np.int32),
                                np.asarray(indptr, np.int64)), shape=(Nd, max(V, 1)))
        self.vocab, self.dl = vocab, dl
        self.avgdl = float(dl.mean()) if Nd else 1.0
        df = np.asarray((self.M > 0).sum(axis=0)).ravel().astype(np.float32)
        self.idf = np.log(1.0 + (Nd - df + 0.5) / (df + 0.5)).astype(np.float32)
        self.denom_doc = (self.k1 * (1 - self.b + self.b * self.dl / max(self.avgdl, 1e-6))).astype(np.float32)
        return self

    def _csc(self):
        if self._Mcsc is None:
            self._Mcsc = self.M.tocsc()
        return self._Mcsc

    def search(self, query_tokens, topk=200):
        if self.M is None or self.M.shape[0] == 0:
            return np.array([], np.int64), np.array([], np.float32)
        qc = Counter(t for t in query_tokens if t in self.vocab)
        if not qc:
            return np.array([], np.int64), np.array([], np.float32)
        scores = np.zeros(self.M.shape[0], dtype=np.float32)
        Mcsc = self._csc()
        for t in qc:
            j = self.vocab[t]
            s, e = Mcsc.indptr[j], Mcsc.indptr[j + 1]
            rows, tf = Mcsc.indices[s:e], Mcsc.data[s:e]
            scores[rows] += self.idf[j] * (tf * (self.k1 + 1)) / (tf + self.denom_doc[rows])
        k = min(topk, len(scores))
        idx = np.argpartition(-scores, k - 1)[:k]
        idx = idx[np.argsort(-scores[idx])]
        idx = idx[scores[idx] > 0]
        return idx.astype(np.int64), scores[idx]

    def save(self, prefix):
        sp.save_npz(prefix + ".npz", self.M)
        jdump({"k1": self.k1, "B": self.b, "avgdl": self.avgdl, "vocab": self.vocab}, prefix + ".vocab.json")
        np.savez(prefix + ".stats.npz", idf=self.idf, dl=self.dl, denom_doc=self.denom_doc)

    @classmethod
    def load(cls, prefix):
        o = cls(); o.M = sp.load_npz(prefix + ".npz")
        v = jload(prefix + ".vocab.json")
        o.k1, o.b, o.avgdl, o.vocab = v["k1"], v["B"], v["avgdl"], v["vocab"]
        st = np.load(prefix + ".stats.npz")
        o.idf, o.dl, o.denom_doc = st["idf"], st["dl"], st["denom_doc"]
        o._Mcsc = None
        return o

# (doc_table, analyzer)
BM25_FIELDS = [
    ("ocr", "raw"), ("ocr", "fold"),
    ("caption", "raw"),
    ("transcript_vi", "raw"), ("transcript_vi", "fold"),
    ("transcript_en", "raw"),
    ("summary", "raw"), ("summary", "fold"),
]

BM25_DIR = ART / "bm25"; BM25_DIR.mkdir(exist_ok=True)
if stage_done("bm25"):
    print("[Stage:bm25] skip")
else:
    for table, ana in BM25_FIELDS:
        df = DOCS[table]
        toks = [tokenize(t, fold_accent=(ana == "fold")) for t in df.text.astype(str).tolist()]
        t0 = time.time()
        bm = BM25(CFG["bm25_k1"], CFG["bm25_b"]).fit(toks)
        bm.save(str(BM25_DIR / ("%s__%s" % (table, ana))))
        print("  bm25 %s__%s: docs=%d vocab=%d %.0fs" % (table, ana, len(toks), len(bm.vocab), time.time() - t0))
    mark_done("bm25", {"fields": ["%s__%s" % (a, b) for a, b in BM25_FIELDS]})

_bm = BM25.load(str(BM25_DIR / "ocr__fold"))
_i, _s = _bm.search(tokenize("thôi sự", fold_accent=True), topk=5)
print("smoke bm25 ocr__fold:", _i.tolist(), np.round(_s, 3).tolist())

## Stage `objects` - Inverted index

Object detection có thể có false negative cao, vì vậy luôn chỉ dùng làm soft boost, không bao giờ làm hard filter.

Notebook lưu `row_id`, `entity`, `score` cùng bảng `entity -> df, idf` để tính rarity boost.

In [11]:
# ============================== CELL 10: Stage objects ==============================
if "objects" not in MOD:
    print("[Stage:objects] Không có Object dir -> skip (Fallback: không dùng Object boost)")
elif stage_done("objects"):
    print("[Stage:objects] skip")
else:
    OBJ = Path(MOD["objects"])
    rows, t0 = [], time.time()
    for vi, vid in enumerate(video_ids):
        d = OBJ / vid
        if not d.is_dir():
            continue
        for jf in d.glob("*.json"):
            try:
                nn = int(jf.stem)
            except ValueError:
                continue
            rid = KF_INDEX.get((vid, nn))
            if rid is None:
                continue
            try:
                o = jload(jf)
            except Exception:
                continue
            ents = o.get("detection_class_entities") or []
            scs = o.get("detection_scores") or []
            seen = {}
            for e, s in zip(ents, scs):
                try:
                    s = float(s)
                except Exception:
                    continue
                if s < CFG["object_conf_threshold"]:
                    continue
                e = str(e).strip().lower()
                if e and s > seen.get(e, 0.0):
                    seen[e] = s
            for e, s in sorted(seen.items(), key=lambda x: -x[1])[: CFG["object_topk_per_frame"]]:
                rows.append((rid, e, round(s, 4)))
        if (vi + 1) % 100 == 0:
            print("  objects %d/%d rows=%d %.0fs" % (vi + 1, len(video_ids), len(rows), time.time() - t0))
    OBJDF = pd.DataFrame(rows, columns=["row_id", "entity", "score"])
    OBJDF.to_parquet(ART / "object_index.parquet", index=False)
    ent = OBJDF.groupby("entity")["row_id"].nunique().rename("df").reset_index()
    ent["idf"] = np.log(1.0 + len(KF) / np.maximum(ent["df"], 1))
    ent.to_parquet(ART / "object_entities.parquet", index=False)
    print("Object rows:", len(OBJDF), "| distinct entities:", len(ent))
    mark_done("objects", {"rows": int(len(OBJDF)), "entities": int(len(ent))})

## Stage `text_embed` - Embedding qua OpenRouter

- Cache content-addressed dùng `sha1(model_slug | version | text)`, nên chạy lại không tốn thêm tiền.
- `MAX_TOTAL_COST_USD=2.00` được dùng chung với NB02 qua `cost_ledger.json`; có cảnh báo ở 80% và dừng ở 90%.
- Retry có giới hạn và exponential backoff; lỗi 4xx do schema/auth không retry.

In [12]:
# ============================== CELL 11: COST LEDGER + OPENROUTER CLIENT ==============================
import urllib.request, urllib.error

LEDGER = ART / "cost_ledger.json"
ERRLOG = ART / "error_ledger.jsonl"

def ledger_read():
    if LEDGER.exists():
        try:
            return jload(LEDGER)
        except Exception:
            pass
    return {"total_usd": 0.0, "by_model": {}, "events": 0}

def ledger_add(model, usd, tokens=0, calls=1):
    L = ledger_read()
    L["total_usd"] = round(float(L["total_usd"]) + float(usd), 6)
    m = L["by_model"].setdefault(model, {"usd": 0.0, "tokens": 0, "calls": 0})
    if "usd" not in m:                         # tương thích ledger cũ bị viết hoa
        m["usd"] = float(m.pop("USD", 0.0))
    m["usd"] = round(m["usd"] + float(usd), 6); m["tokens"] += int(tokens); m["calls"] += int(calls)
    L["events"] += 1
    jdump(L, LEDGER)
    return L

def budget_check(extra_usd=0.0):
    L = ledger_read()
    tot = L["total_usd"] + extra_usd
    cap = CFG["MAX_TOTAL_COST_USD"]
    if tot >= cap * CFG["COST_HALT_RATIO"]:
        raise RuntimeError("[COST HALT] %.4f USD >= %d%% của CAP %.2f USD"
                           % (tot, CFG["COST_HALT_RATIO"] * 100, cap))
    if tot >= cap * CFG["COST_WARN_RATIO"]:
        print("[COST WARN] %.4f/%.2f USD (%.0f%%)" % (tot, cap, 100 * tot / cap))
    return tot

def log_error(kind, detail):
    with open(ERRLOG, "a", encoding="UTF-8") as f:
        f.write(json.dumps({"ts": time.time(), "kind": kind, "detail": str(detail)[:1500]},
                           ensure_ascii=False) + "\n")

def approx_tokens(s):
    """Ước lượng token: ~4 char/token cho EN; tiếng Việt đây hơn -> dùng 3.2."""
    return max(1, int(len(s) / 3.2))

def or_post(path, payload, timeout=None, retries=None):
    """POST OpenRouter, Retry hữu hạn + exponential backoff. Không log API key."""
    assert OPENROUTER_API_KEY, "Thiếu OPENROUTER_API_KEY (env hoặc Kaggle Secret)"
    url = CFG["openrouter_base"].rstrip("/") + path
    body = json.dumps(payload).encode("UTF-8")
    retries = CFG["embed_max_retries"] if retries is None else retries
    timeout = CFG["embed_timeout"] if timeout is None else timeout
    last = None
    for a in range(retries):
        req = urllib.request.Request(url, data=body, method="POST", headers={
            "Authorization": "Bearer " + OPENROUTER_API_KEY,
            "Content-Type": "application/json",
            "HTTP-Referer": "https://kaggle.com", "X-Title": "aic2026",
        })
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode("UTF-8"))
        except urllib.error.HTTPError as e:
            msg = e.read().decode("UTF-8", "replace")[:600]
            last = "HTTP %d: %s" % (e.code, msg)
            log_error("HTTP", last)
            if e.code in (400, 401, 403, 404, 422):
                raise RuntimeError("[non-retryable] " + last)   # lỗi schema/auth: không Retry vo hạn
        except Exception as e:
            last = type(e).__name__ + ": " + str(e); log_error("net", last)
        sleep = min(30, 2 ** a) + 0.3 * a
        print("  Retry %d/%d sau %.1fs (%s)" % (a + 1, retries, sleep, str(last)[:120]))
        time.sleep(sleep)
    raise RuntimeError("OpenRouter thất bại sau %d lần: %s" % (retries, last))

print("ledger:", ledger_read())

In [13]:
# ============================== CELL 12: Stage text_embed ==============================
EMB_DIR = ART / "text_embed"; EMB_DIR.mkdir(exist_ok=True)
ECACHE = CACHE / "embed"

def cache_key(text):
    return sha1(CFG["embed_model"] + "|v1|" + text)

def cache_path(key):
    return ECACHE / key[:2] / (key + ".npy")

def cache_get(text):
    p = cache_path(cache_key(text))
    if p.exists():
        try:
            return np.load(p)
        except Exception:
            return None
    return None

def cache_put(text, vec):
    p = cache_path(cache_key(text)); p.parent.mkdir(parents=True, exist_ok=True)
    np.save(p, np.asarray(vec, dtype=np.float32))

def embed_texts(texts, tag=""):
    """Trả về (N, embed_dim) float32 L2-normalized. Dùng Cache, tốn trong DRY_RUN và budget."""
    N = len(texts)
    out = np.zeros((N, CFG["embed_dim"]), dtype=np.float32)
    todo = []
    for i, t in enumerate(texts):
        v = cache_get(t)
        if v is not None:
            out[i] = v
        else:
            todo.append(i)
    tok = sum(approx_tokens(texts[i]) for i in todo)
    est = tok / 1e6 * CFG["price_embed_per_1m"]
    print("[embed:%s] docs=%d cached=%d todo=%d est_tokens=%s est_cost=$%.4f"
          % (tag, N, N - len(todo), len(todo), human(tok), est))
    if CFG["DRY_RUN"]:
        print("[embed:%s] DRY_RUN -> không gọi API" % tag)
        return None, {"docs": N, "todo": len(todo), "est_tokens": tok, "est_cost": est, "dry_run": True}
    if not todo:
        return out, {"docs": N, "todo": 0, "est_cost": 0.0}
    budget_check(est)
    spent, B = 0.0, CFG["embed_batch"]
    for s in range(0, len(todo), B):
        chunk = todo[s:s + B]
        payload = {"model": CFG["embed_model"], "input": [texts[i][:8000] for i in chunk]}
        try:
            res = or_post("/embeddings", payload)
        except RuntimeError as e:
            log_error("embed_batch_fail", "%s %d: %s" % (tag, s, e))
            print("  [skip batch]", e)   # đọc này thiếu vector, Không làm chết ca Stage
            continue
        data = res.get("data", [])
        if len(data) != len(chunk):
            log_error("embed_len_mismatch", "%s: %d vs %d" % (tag, len(data), len(chunk)))
        for d, i in zip(data, chunk):
            v = np.asarray(d.get("embedding", []), dtype=np.float32)
            if v.shape[0] != CFG["embed_dim"]:
                log_error("embed_dim", "got %s" % (v.shape,)); continue
            v = v / max(float(np.linalg.norm(v)), 1e-8)
            out[i] = v; cache_put(texts[i], v)
        u = res.get("usage") or {}
        tk = int(u.get("prompt_tokens", u.get("total_tokens", 0)) or sum(approx_tokens(texts[i]) for i in chunk))
        c = tk / 1e6 * CFG["price_embed_per_1m"]; spent += c
        ledger_add(CFG["embed_model"], c, tk, 1)
        if spent > CFG["embed_budget_usd"]:
            print("  [embed budget stop] đã chỉ $%.4f > trần Stage $%.2f" % (spent, CFG["embed_budget_usd"]))
            break
        if (s // B) % 20 == 0:
            print("  %d/%d | spent $%.4f | ledger $%.4f"
                  % (s + len(chunk), len(todo), spent, ledger_read()["total_usd"]))
    return out, {"docs": N, "todo": len(todo), "spent": spent}

EMBED_TABLES = [k for k, v in CFG["embed_targets"].items() if v]
embed_report = {}
ROW_COLS = ["doc_id", "video_id", "keyframe_n", "frame_idx", "pts_time", "start_time", "end_time", "modality"]
for table in EMBED_TABLES:
    fp = EMB_DIR / ("emb_%s.npy" % table)
    if fp.exists() and "text_embed" not in CFG["force_stages"]:
        print("  loaded emb_%s: %s" % (table, np.load(fp, mmap_mode="r").shape)); continue
    df = DOCS[table]
    vecs, rep = embed_texts(df.text.astype(str).tolist(), tag=table)
    embed_report[table] = rep
    if vecs is not None:
        np.save(fp, vecs.astype(np.float32))
        df[ROW_COLS].to_parquet(EMB_DIR / ("rows_%s.parquet" % table), index=False)
        print("  saved emb_%s: %s" % (table, vecs.shape))

jdump(embed_report, ART / "embed_dryrun_report.json")
print("\n=== COST ESTIMATE ===")
tot = sum(r.get("est_cost", 0.0) for r in embed_report.values())
for k, r in embed_report.items():
    print("  %-14s docs=%7d tokens=%8s est=$%.4f"
          % (k, r.get("docs", 0), human(r.get("est_tokens", 0)), r.get("est_cost", 0.0)))
print("  TOTAL embedding one-Off: $%.4f (trần Stage $%.2f, trần tổng $%.2f)"
      % (tot, CFG["embed_budget_usd"], CFG["MAX_TOTAL_COST_USD"]))
print("\nDe chạy thật: đặt CFG['DRY_RUN']=False Rồi chạy lại CELL 12.")

In [14]:
# ============================== CELL 13: Stage text_faiss ==============================
# Chỉ Build Khi đã có File embedding thực (không phải DRY-run).
if not HAS_FAISS:
    print("[Stage:text_faiss] thiếu FAISS -> skip")
else:
    built = {}
    for table in EMBED_TABLES:
        fp = EMB_DIR / ("emb_%s.npy" % table)
        ifp = ART / ("faiss_text_%s.index" % table)
        if not fp.exists():
            print("  [skip] Chưa có emb_%s.npy (đang DRY_RUN?)" % table); continue
        if ifp.exists() and "text_faiss" not in CFG["force_stages"]:
            print("  [skip] đã có", ifp.name); continue
        X = np.load(fp).astype(np.float32)
        keep = np.linalg.norm(X, axis=1) > 1e-6      # bộ đọc thiếu vector (batch lỗi)
        idx_map = np.where(keep)[0].astype(np.int64)
        Xk = X[keep]
        Xk /= np.clip(np.linalg.norm(Xk, axis=1, keepdims=True), 1e-8, None)
        ix = faiss.IndexFlatIP(Xk.shape[1]); ix.add(Xk)
        faiss.write_index(ix, str(ifp))
        np.save(ART / ("faiss_text_%s_rowmap.npy" % table), idx_map)  # faiss_row -> Row trong docs_<table>
        built[table] = {"ntotal": int(ix.ntotal), "dropped": int((~keep).sum())}
        print("  faiss_text_%s: ntotal=%d dropped=%d" % (table, ix.ntotal, int((~keep).sum())))
    if built:
        mark_done("text_faiss", built)

## Stage `manifest` - Kiểm tra toàn vẹn và bàn giao cho NB02

`artifact_manifest.json` là hợp đồng dữ liệu duy nhất mà NB02 được phép sử dụng. Nếu thiếu artifact, NB02 phải degrade nhánh tương ứng chứ không được crash.

In [15]:
# ============================== CELL 14: Integrity check + MANIFEST ==============================
def fsize(p):
    p = Path(p)
    return int(p.stat().st_size) if p.exists() else 0

checks, problems = {}, []

# C1: Row alignment giữa FAISS Visual và keyframes.parquet
if (ART / "faiss_siglip2.index").exists() and HAS_FAISS:
    _ix = faiss.read_index(str(ART / "faiss_siglip2.index"))
    checks["visual_ntotal_eq_keyframes"] = (_ix.ntotal == len(KF))
    if _ix.ntotal != len(KF):
        problems.append("P0 Visual Index ntotal %d != keyframes %d" % (_ix.ntotal, len(KF)))
else:
    problems.append("P0 thiếu faiss_siglip2.index")

# C2: mỗi đọc phải trỏ về Video tồn tại
vid_set = set(KF.video_id.unique())
for name, df in DOCS.items():
    bad = int((~df.video_id.isin(vid_set)).sum())
    checks["docs_%s_video_ok" % name] = (bad == 0)
    if bad:
        problems.append("P1 docs_%s: %d đọc trỏ về Video không có trong keyframes" % (name, bad))

# C3: frame_idx phải tăng theo keyframe_n trong từng Video
mono = KF.sort_values(["video_id", "keyframe_n"]).groupby("video_id")["frame_idx"] \
         .apply(lambda s: bool(s.is_monotonic_increasing))
checks["frame_idx_monotonic"] = bool(mono.all())
if not mono.all():
    problems.append("P1 frame_idx không monotonic ở %d Video" % int((~mono).sum()))

# C4: coverage tối thiểu
checks["coverage_caption_ge_0.8"] = bool(audit["has_caption"].mean() >= 0.8)
checks["coverage_asr_en_ge_0.8"] = bool(audit["has_asr_en"].mean() >= 0.8)

MANIFEST = {
    "created_ts": time.time(),
    "Notebook": "01_build_indices_api",
    "config": {k: v for k, v in CFG.items() if k != "force_stages"},
    "discovered": {"modalities": MOD, "n_video_files": len(VIDEO_FILES),
                   "n_keyframe_dirs": len(KEYFRAME_DIRS), "query_dir": QUERY_DIR},
    "video_files": VIDEO_FILES,
    "keyframe_dirs": KEYFRAME_DIRS,
    "video_ids": video_ids,
    "counts": dict({"videos": len(video_ids), "keyframe_rows": int(len(KF))},
                   **{"docs_" + k: int(len(v)) for k, v in DOCS.items()}),
    "artifacts": {
        "keyframes":      {"path": str(ART / "keyframes.parquet"), "bytes": fsize(ART / "keyframes.parquet")},
        "Audit":          {"path": str(ART / "audit_videos.parquet"), "bytes": fsize(ART / "audit_videos.parquet")},
        "faiss_siglip2":  {"path": str(ART / "faiss_siglip2.index"), "bytes": fsize(ART / "faiss_siglip2.index"),
                           "dim": CFG["siglip_dim"], "Metric": "IP (cosine trên vector L2-normed)",
                           "Model": "google/siglip2-giant-opt-patch16-384"},
        "siglip2_matrix": {"path": str(ART / "siglip2_matrix_f16.npy"), "bytes": fsize(ART / "siglip2_matrix_f16.npy")},
        "bm25_dir":       {"path": str(BM25_DIR), "fields": ["%s__%s" % (a, b) for a, b in BM25_FIELDS]},
        "object_index":   {"path": str(ART / "object_index.parquet"), "bytes": fsize(ART / "object_index.parquet"),
                           "conf_threshold": CFG["object_conf_threshold"], "usage": "SOFT BOOST ONLY"},
        "text_embed_dir": {"path": str(EMB_DIR), "Model": CFG["embed_model"], "dim": CFG["embed_dim"],
                           "tables": {t: {"emb_bytes": fsize(EMB_DIR / ("emb_%s.npy" % t)),
                                          "faiss_bytes": fsize(ART / ("faiss_text_%s.index" % t))}
                                      for t in EMBED_TABLES}},
        "docs":           {k: str(ART / ("docs_%s.parquet" % k)) for k in DOCS},
        "cost_ledger":    {"path": str(LEDGER), "state": ledger_read()},
    },
    "Integrity": checks,
    "problems": problems,
    "notes": [
        "SigLIP2 và text-embedding-3-small là 2 không giản khác nhau -> Không báo giờ trộn vector.",
        "object_index chỉ dùng làm soft boost; không được loại candidate vì thiếu Object.",
        "row_id trong keyframes.parquet == Row id trong faiss_siglip2.index.",
        "faiss_text_<T>_rowmap.npy map FAISS Row -> Row trong docs_<T>.parquet.",
    ],
}
jdump(MANIFEST, ART / "artifact_manifest.json")

print("===== Integrity =====")
for k, v in checks.items():
    print(("  PASS " if v else "  FAIL ") + k)
print("\nproblems:", len(problems))
for p in problems:
    print("  -", p)
n_p0 = sum(1 for p in problems if p.startswith("P0"))
print("\nP0 còn lại:", n_p0)
print("manifest ->", ART / "artifact_manifest.json")
if n_p0 == 0:
    mark_done("manifest", {"checks": checks})
    print("\nNB01 READY -> chuyển sang 02_retrieve_refine_candidates_api.ipynb")
else:
    print("\n[BLOCK] con lỗi P0, không được chuyển sang NB02")

In [16]:
# ============================== CELL 15: Smoke test Tổng ==============================
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("  PASS " if cond else "  FAIL ") + name + (("  (" + detail + ")") if detail else ""))
    ok = ok and bool(cond)

man = jload(ART / "artifact_manifest.json")
chk("manifest readable", isinstance(man, dict), str(list(man.keys())[:4]))
chk("keyframes.parquet", (ART / "keyframes.parquet").exists(), "%d rows" % len(KF))
chk("FAISS Visual", (ART / "faiss_siglip2.index").exists())
chk("bm25 fields", all((BM25_DIR / ("%s__%s.npz" % (a, b))).exists() for a, b in BM25_FIELDS))
chk("docs tables", all((ART / ("docs_%s.parquet" % k)).exists() for k in DOCS))
chk("COST dưới CAP", ledger_read()["total_usd"] < CFG["MAX_TOTAL_COST_USD"],
    "$%.4f" % ledger_read()["total_usd"])

# duyệt ngược: lấy 1 Keyframe ngẫu nhiên, tìm lại mỗi modality
r = KF.sample(1, random_state=7).iloc[0]
print("\nspot-check %s N=%d frame_idx=%d T=%.2fs" % (r.video_id, r.keyframe_n, r.frame_idx, r.pts_time))
for k, df in DOCS.items():
    sub = df[(df.video_id == r.video_id) & (df.keyframe_n == r.keyframe_n)]
    if not len(sub):
        sub = df[df.video_id == r.video_id].head(1)
    if len(sub):
        print("  %-14s: %r" % (k, str(sub.text.iloc[0])[:90]))
print("\nSMOKE:", "OK" if ok else "FAILED")
print("\nNext: 02_retrieve_refine_candidates_api.ipynb")